# Test if pesticide usage is a useful predictor

Compare baseline model to models that have the baseline features and:

1. feature engineered 2 (respiratory) and 6 (chemical class aggregate) pesticide features
2. 30+ features that survive Lasso regression
3. principle components that explain 90% of the variance
4. all 400+ individual pesticide compounds

In [3]:
import os
import numpy as np
import pandas as pd

import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt


In [4]:
DATA_DIR = "../data"

TRAIN_CASTHMA_PATH = os.path.join(DATA_DIR, "train_CASTHMA.csv")

TRAIN_COPD_PATH = os.path.join(DATA_DIR, "train_COPD.csv")

train_asthma = pd.read_csv(TRAIN_CASTHMA_PATH)
train_copd = pd.read_csv(TRAIN_COPD_PATH)

In [22]:
# Shared non-pesticide features across all models
DEMO_COLS = [
    'population', 'median_age', 'median_income',
    'pct_white', 'pct_black', 'pct_asian', 'pct_hispanic',
    'rural_binary',   # binary: 0=urban, 1=rural
]
HEALTH_CONFOUNDER_COLS = ['CSMOKING', 'OBESITY', 'DIABETES']
CROPLAND_COLS = ['cropland_diversity', 'county_crop_concentration', 'pct_cropland']
OTHER_COLS = ['YEAR']

BASE_COLS = DEMO_COLS + HEALTH_CONFOUNDER_COLS + CROPLAND_COLS + OTHER_COLS

# Pesticide set A: respiratory aggregate (log-transformed intensity)
PEST_AGGREGATE = [
    'log_resp_per_capita',
    'log_resp_per_cropland_acre',
]

# Pesticide set B: individual components (log-transformed intensity)
# OP, carbamate, pyrethroid separately — never mixed with aggregate
PEST_COMPONENTS = [
    'log_op_per_capita',
    'log_op_per_cropland_acre',
    'log_carbamate_per_capita',
    'log_carbamate_per_cropland_acre',
    'log_pyrethroid_per_capita',
    'log_pyrethroid_per_cropland_acre',
]

FEATURES_AGGREGATE   = PEST_AGGREGATE + BASE_COLS
FEATURES_COMPONENTS  = PEST_COMPONENTS + BASE_COLS

BASE_FEATURES = BASE_COLS
EXT_FEATURES = BASE_COLS + PEST_AGGREGATE

print(f"Aggregate feature set:  {len(FEATURES_AGGREGATE)} features")
print(f"Components feature set: {len(FEATURES_COMPONENTS)} features")
print("\nAggregate pesticide features:", PEST_AGGREGATE)
print("Component pesticide features:", PEST_COMPONENTS)

Aggregate feature set:  17 features
Components feature set: 21 features

Aggregate pesticide features: ['log_resp_per_capita', 'log_resp_per_cropland_acre']
Component pesticide features: ['log_op_per_capita', 'log_op_per_cropland_acre', 'log_carbamate_per_capita', 'log_carbamate_per_cropland_acre', 'log_pyrethroid_per_capita', 'log_pyrethroid_per_cropland_acre']


In [ ]:
DEMO_COLS = [
    'population','median_age','median_income',
    'pct_white','pct_black','pct_asian','pct_hispanic',
    'rural_binary']
HEALTH_CONFOUNDER_COLS = ['CSMOKING','OBESITY','DIABETES']
CROPLAND_COLS = ['cropland_diversity','county_crop_concentration','pct_cropland']
OTHER_COLS = ['YEAR']

BASE_COLS = DEMO_COLS + HEALTH_CONFOUNDER_COLS + CROPLAND_COLS + OTHER_COLS

PEST_AGGREGATE = [
    'log_resp_per_capita',
    'log_resp_per_cropland_acre']

PEST_COMPONENTS = [
    'log_op_per_capita',
    'log_op_per_cropland_acre',
    'log_carbamate_per_capita',
    'log_carbamate_per_cropland_acre',
    'log_pyrethroid_per_capita',
    'log_pyrethroid_per_cropland_acre']

cols = train_asthma.columns[train_asthma.columns.str.startswith("pesticide_")]
cols_to_drop = ['pesticide_total_kg','pesticide_respiratory_kg',
                 'pesticide_organochlorine_kg','pesticide_organophosphate_kg',
                 'pesticide_carbamate_kg','pesticide_pyrethroid_kg',
                 'pesticide_triazine_kg','pesticide_chlorophenoxy_kg',
                 'pesticide_anilide_kg','pesticide_other_kg']
PEST_ALL = [f'{col}' for col in cols if col not in cols_to_drop]
PEST_ALL_CAP = [f'log_{col[10:-3]}_per_cap' for col in cols if col not in cols_to_drop] # extracted 400+ individual component list
PEST_ALL_CROP = [f'log_{col[10:-3]}_per_crop' for col in cols if col not in cols_to_drop] # extracted 400+ individual component list

In [6]:
def engineer_features(df):
    df = df.copy()

    df = df.fillna(0)

    df['resp_per_capita'] = df['pesticide_respiratory_kg'] / df['population']
    df['resp_per_cropland_acre'] = df['pesticide_respiratory_kg'] / df['cropland_acres']

    df['op_per_capita'] = df['pesticide_organophosphate_kg'] / df['population']
    df['op_per_cropland_acre'] = df['pesticide_organophosphate_kg'] / df['cropland_acres']

    df['carbamate_per_capita'] = df['pesticide_carbamate_kg'] / df['population']
    df['carbamate_per_cropland_acre'] = df['pesticide_carbamate_kg'] / df['cropland_acres']

    df['pyrethroid_per_capita'] = df['pesticide_pyrethroid_kg'] / df['population']
    df['pyrethroid_per_cropland_acre'] = df['pesticide_pyrethroid_kg'] / df['cropland_acres']

    pest_intensity_cols = [
        'resp_per_capita','resp_per_cropland_acre',
        'op_per_capita', 'op_per_cropland_acre',
        'carbamate_per_capita', 'carbamate_per_cropland_acre',
        'pyrethroid_per_capita', 'pyrethroid_per_cropland_acre']
    for col in pest_intensity_cols:
        df[f'log_{col}'] = np.log1p(df[col])

    df['rural_binary'] = (df['nchs_urban_rural'] >= 5).astype(int)

    # take all pesticide columns excluding the aggregate ones and normalize the raw component kg
    cols = df.columns[df.columns.str.startswith("pesticide_")]
    cols_to_drop = ['pesticide_total_kg','pesticide_respiratory_kg',
                     'pesticide_organochlorine_kg','pesticide_organophosphate_kg',
                     'pesticide_carbamate_kg','pesticide_pyrethroid_kg',
                     'pesticide_triazine_kg','pesticide_chlorophenoxy_kg',
                     'pesticide_anilide_kg']
    cols = [col for col in cols if col not in cols_to_drop] # extracted 400+ individual component list
    for col in cols:
        df[f'log_{col[10:-3]}_per_cap'] = np.log1p(df[col] / df['population'])
        df[f'log_{col[10:-3]}_per_crop'] = np.log1p(df[col] / df['cropland_acres'])
    
    return df
    
train_asthma = engineer_features(train_asthma)
train_copd = engineer_features(train_copd)
    

/var/folders/p8/78zqnzm55rb_1dr0p6h3v3bm0000gn/T/ipykernel_16528/3928423203.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'log_{col[10:-3]}_per_crop'] = np.log1p(df[col] / df['cropland_acres'])
/var/folders/p8/78zqnzm55rb_1dr0p6h3v3bm0000gn/T/ipykernel_16528/3928423203.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'log_{col[10:-3]}_per_cap'] = np.log1p(df[col] / df['population'])
/var/folders/p8/78zqnzm55rb_1dr0p6h3v3bm0000gn/T/ipykernel_16528/3928423203.py:38: PerformanceWarning: DataFrame is highly fragme

In [8]:
def compare_models(df, target, base_predictors, extra_predictors):

    base_formula = target + " ~ " + " + ".join(base_predictors)
    full_formula = target + " ~ " + " + ".join(base_predictors + extra_predictors)

    base_model = smf.ols(base_formula, data=df).fit()
    full_model = smf.ols(full_formula, data=df).fit()

    anova = sm.stats.anova_lm(base_model, full_model)

    # Perform the F-test
    f_test = full_model.compare_f_test(base_model)
    
    # Print the results
    print("F-statistic:", f_test[0])
    print("p-value:", f_test[1])
    if f_test[1] < 0.05:
        print("Full model improves fit")

    adj_rsquared_change=full_model.rsquared_adj-base_model.rsquared_adj
    print(f"Adjusted R squared: {np.round(base_model.rsquared_adj,3)} -> {np.round(full_model.rsquared_adj,3)} ({np.round(adj_rsquared_change,3)})")

    return {
        "base_model": base_model,
        "full_model": full_model,
        "anova": anova,
        "F": f_test[0],
        "p": f_test[1],
        "aic": (base_model.aic, full_model.aic),
        "bic": (base_model.bic, full_model.bic),
        "adj_rsquared_change":adj_rsquared_change
    }

### 1. Baseline + engineered features for pesticides

In [9]:
base_vs_aggregate_asthma = compare_models(
    train_asthma,
    target="CASTHMA",
    base_predictors=BASE_COLS,
    extra_predictors=PEST_AGGREGATE
)
base_vs_component_asthma = compare_models(
    train_asthma,
    target="CASTHMA",
    base_predictors=BASE_COLS,
    extra_predictors=PEST_AGGREGATE
)

base_vs_aggregate_copd = compare_models(
    train_copd,
    target="COPD",
    base_predictors=BASE_COLS,
    extra_predictors=PEST_COMPONENTS
)
base_vs_component_copd = compare_models(
    train_copd,
    target="COPD",
    base_predictors=BASE_COLS,
    extra_predictors=PEST_COMPONENTS
)

F-statistic: 57.37931120904318
p-value: 2.347215875535327e-25
Full model improves fit
Adjusted R squared: 0.63 -> 0.638 (0.008)
F-statistic: 57.37931120904318
p-value: 2.347215875535327e-25
Full model improves fit
Adjusted R squared: 0.63 -> 0.638 (0.008)
F-statistic: 18.669661787922255
p-value: 1.3931534381562201e-21
Full model improves fit
Adjusted R squared: 0.853 -> 0.856 (0.003)
F-statistic: 18.669661787922255
p-value: 1.3931534381562201e-21
Full model improves fit
Adjusted R squared: 0.853 -> 0.856 (0.003)


### 2. Baseline + Lasso-surviving features
Testing only on pesticide per capita for now

In [10]:
# run lasso regression and then select the pesticide-features that survive

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.pipeline import Pipeline

def choose_lasso_features(X,y):

    lasso_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", LassoCV(cv=5, random_state=42))
    ])
    
    lasso_pipe.fit(X, train_asthma['CASTHMA'])
    
    lasso_model = lasso_pipe.named_steps["lasso"]
    
    coef = pd.Series(lasso_model.coef_, index=X.columns)
    
    selected_features = coef[coef != 0].index.tolist()

    return selected_features

In [11]:
selected_features = choose_lasso_features(train_asthma[BASE_COLS+PEST_ALL_CAP],train_asthma['CASTHMA'])
lasso_pest_asthma = [c for c in selected_features if c.startswith("log_")]
len(lasso_pest_asthma)

/Users/Matthew/Desktop/spring-2026-pesticide-exposure/venv/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.870e-01, tolerance: 4.014e-01
  model = cd_fast.enet_coordinate_descent_gram(
/Users/Matthew/Desktop/spring-2026-pesticide-exposure/venv/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.597e-01, tolerance: 3.695e-01
  model = cd_fast.enet_coordinate_descent_gram(
/Users/Matthew/Desktop/spring-2026-pesticide-exposure/venv/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might w

116

In [12]:
base_vs_lasso_asthma = compare_models(
    train_asthma,
    target="CASTHMA",
    base_predictors=BASE_COLS,
    extra_predictors=lasso_pest_asthma
)

F-statistic: 13.858961652456998
p-value: 1.1699995024952289e-217
Full model improves fit
Adjusted R squared: 0.63 -> 0.716 (0.086)


In [10]:
selected_features = choose_lasso_features(train_copd[BASE_COLS+PEST_ALL_CAP],train_copd['COPD'])
lasso_pest_copd = [c for c in selected_features if c.startswith("log_")]
len(lasso_pest_copd)

/home/sunyoung/miniconda3/envs/pesticide/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.451e-01, tolerance: 4.014e-01
  model = cd_fast.enet_coordinate_descent_gram(
/home/sunyoung/miniconda3/envs/pesticide/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.202e+00, tolerance: 4.014e-01
  model = cd_fast.enet_coordinate_descent_gram(
/home/sunyoung/miniconda3/envs/pesticide/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the sc

27

In [11]:
base_vs_lasso_copd = compare_models(
    train_copd,
    target="COPD",
    base_predictors=BASE_COLS,
    extra_predictors=lasso_pest_copd
)

F-statistic: 12.027302181010045
p-value: 7.061692332420416e-51
Full model improves fit
Adjusted R squared: 0.853 -> 0.861 (0.009)


### 3. Baseline + PCA on pesticides
Testing only on pesticide per capita for now

In [13]:
def add_pcs(df):
    df=df.copy()
    X=df[PEST_ALL_CAP]
    
    # Scale
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # PCA
    pca = PCA(n_components=0.9) # cutoff at 90% explained variance
    X_pca = pca.fit_transform(X_scaled)
    X_pca = pd.DataFrame(X_pca)
    X_pca = X_pca.add_prefix('PC_')
    
    df = pd.concat([df,X_pca],axis=1)
    cols = X_pca.columns.tolist()
    
    return df, cols

In [14]:
train_asthma_PCA, PC_COLS = add_pcs(train_asthma)

base_vs_PCs_asthma = compare_models(
    train_asthma_PCA,
    target="CASTHMA",
    base_predictors=BASE_COLS,
    extra_predictors=PC_COLS
)

F-statistic: 8.075620985783303
p-value: 6.044650327552564e-100
Full model improves fit
Adjusted R squared: 0.63 -> 0.677 (0.047)


In [15]:
train_copd_PCA, PC_COLS = add_pcs(train_copd)
base_vs_PCs_copd = compare_models(
    train_copd_PCA,
    target="COPD",
    base_predictors=BASE_COLS,
    extra_predictors=PC_COLS
)

F-statistic: 4.876182858429598
p-value: 1.5326612102123697e-49
Full model improves fit
Adjusted R squared: 0.853 -> 0.864 (0.011)


### 4. Baseline + 400+ pesticides

In [16]:
base_vs_all_cap_asthma = compare_models(
    train_asthma,
    target="CASTHMA",
    base_predictors=BASE_COLS,
    extra_predictors=PEST_ALL_CAP
)
base_vs_all_crop_asthma = compare_models(
    train_asthma,
    target="CASTHMA",
    base_predictors=BASE_COLS,
    extra_predictors=PEST_ALL_CROP
)

base_vs_all_cap_copd = compare_models(
    train_copd,
    target="COPD",
    base_predictors=BASE_COLS,
    extra_predictors=PEST_ALL_CROP
)
base_vs_all_crop_copd = compare_models(
    train_copd,
    target="COPD",
    base_predictors=BASE_COLS,
    extra_predictors=PEST_ALL_CROP
)

F-statistic: 6.047125743464187
p-value: 2.8271427332731536e-197
Full model improves fit
Adjusted R squared: 0.63 -> 0.731 (0.101)
F-statistic: 6.618636759682016
p-value: 1.284397487511768e-223
Full model improves fit
Adjusted R squared: 0.63 -> 0.739 (0.109)
F-statistic: 3.2226772561774983
p-value: 1.7869344357876936e-73
Full model improves fit
Adjusted R squared: 0.853 -> 0.874 (0.021)
F-statistic: 3.2226772561774983
p-value: 1.7869344357876936e-73
Full model improves fit
Adjusted R squared: 0.853 -> 0.874 (0.021)


### 5. Paired t-test on CV Performance

In [40]:
from sklearn.impute import SimpleImputer
def prepare_xy(train_df, target_col, feature_cols):
    """Prepare feature matrix and target from training data only.
    No test data — CV evaluation only.
    Imputer fit on full training set (leakage-safe since we're not touching test).
    """
    feature_cols = [c for c in feature_cols if c in train_df.columns]
    missing = [c for c in feature_cols if c not in train_df.columns]
    if missing:
        print(f"WARNING: missing columns: {missing}")

    X = train_df[feature_cols].copy()
    y = train_df[target_col].copy()

    imputer = SimpleImputer(strategy='median')
    X = pd.DataFrame(imputer.fit_transform(X), columns=feature_cols)

   # Align groups/strat EXACTLY with X
    groups = train_df.loc[X.index, 'FIPS'].values
    strat  = train_df.loc[X.index, f'cat3_{target_col}'].values

    print(f"X: {X.shape}, y nulls: {y.isna().sum()}")
    return X, y, groups, strat, feature_cols

print("--- CASTHMA ---")
X_agg_a,  y_a, grp_a, str_a, fcols_agg_a  = prepare_xy(train_asthma, 'CASTHMA', FEATURES_AGGREGATE)
X_comp_a, y_a, grp_a, str_a, fcols_comp_a = prepare_xy(train_asthma, 'CASTHMA', FEATURES_COMPONENTS)

print("\n--- COPD ---")
X_agg_c,  y_c, grp_c, str_c, fcols_agg_c  = prepare_xy(train_copd, 'COPD', FEATURES_AGGREGATE)
X_comp_c, y_c, grp_c, str_c, fcols_comp_c = prepare_xy(train_copd, 'COPD', FEATURES_COMPONENTS)

--- CASTHMA ---
X: (4871, 17), y nulls: 0
X: (4871, 21), y nulls: 0

--- COPD ---
X: (4871, 17), y nulls: 0
X: (4871, 21), y nulls: 0


In [41]:
# ===== BUILD BASE vs EXT DATASETS =====

print("--- BASE vs EXT: ASTHMA ---")
X_base_a, y_a, grp_a, str_a, _ = prepare_xy(train_asthma, 'CASTHMA', BASE_FEATURES)
X_ext_a,  y_a, grp_a, str_a, _ = prepare_xy(train_asthma, 'CASTHMA', EXT_FEATURES)

print("\n--- BASE vs EXT: COPD ---")
X_base_c, y_c, grp_c, str_c, _ = prepare_xy(train_copd, 'COPD', BASE_FEATURES)
X_ext_c,  y_c, grp_c, str_c, _ = prepare_xy(train_copd, 'COPD', EXT_FEATURES)

--- BASE vs EXT: ASTHMA ---
X: (4871, 15), y nulls: 0
X: (4871, 17), y nulls: 0

--- BASE vs EXT: COPD ---
X: (4871, 15), y nulls: 0
X: (4871, 17), y nulls: 0


In [42]:
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LassoCV, RidgeCV

DATA_DIR = "../data"
RANDOM_STATE = 42
N_SPLITS = 4

def get_cv_splits(X, strat, groups):
    cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    return list(cv.split(X, strat, groups))

def train_and_cv(model_type, X, y, groups, strat, feature_cols):
    """Train model and return CV performance + fitted pipeline.
    Evaluates on cross-validation folds only — no test set.
    """
    cv_splits = get_cv_splits(X, strat, groups)

    if model_type == 'lasso':
        reg = LassoCV(cv=cv_splits, max_iter=10000,
                      random_state=RANDOM_STATE, n_jobs=-1)
        step = 'lasso'
    else:
        reg = RidgeCV(cv=cv_splits)
        step = 'ridge'

    pipe = Pipeline([('scaler', StandardScaler()), (step, reg)])

    # Cross-validate for performance estimate
    r2_scores   = []
    rmse_scores = []
    for train_idx, val_idx in cv_splits:
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        pipe_fold = Pipeline([('scaler', StandardScaler()), (step, reg.__class__(
            max_iter=10000, random_state=RANDOM_STATE, n_jobs=-1
        ) if model_type == 'lasso' else reg.__class__())])
        pipe_fold.fit(X_tr, y_tr)
        y_pred = pipe_fold.predict(X_val)
        r2_scores.append(r2_score(y_val, y_pred))
        rmse_scores.append(np.sqrt(mean_squared_error(y_val, y_pred)))

    # Fit final model on full training data for coefficient inspection
    pipe.fit(X, y)
    best_alpha = pipe.named_steps[step].alpha_
    coefs = pipe.named_steps[step].coef_
    n_nonzero = (coefs != 0).sum()

    results = {
        'cv_r2_mean':   np.mean(r2_scores),
        'cv_r2_std':    np.std(r2_scores),
        'cv_rmse_mean': np.mean(rmse_scores),
        'cv_rmse_std':  np.std(rmse_scores),
        'best_alpha':   best_alpha,
        'n_nonzero':    n_nonzero,
        'n_features':   len(feature_cols),
        'coef_df':      pd.DataFrame({'feature': feature_cols, 'coefficient': coefs})
                          .sort_values('coefficient', key=abs, ascending=False),
    }
    return pipe, results

print("Training all 8 models (4 configs × 2 targets)...")
print("\nModel 1 — Lasso, aggregate, CASTHMA")
m1_a, r1_a = train_and_cv('lasso', X_agg_a,  y_a, grp_a, str_a, fcols_agg_a)
print("Model 1 — Lasso, aggregate, COPD")
m1_c, r1_c = train_and_cv('lasso', X_agg_c,  y_c, grp_c, str_c, fcols_agg_c)

print("\nModel 2 — Ridge, aggregate, CASTHMA")
m2_a, r2_a = train_and_cv('ridge', X_agg_a,  y_a, grp_a, str_a, fcols_agg_a)
print("Model 2 — Ridge, aggregate, COPD")
m2_c, r2_c = train_and_cv('ridge', X_agg_c,  y_c, grp_c, str_c, fcols_agg_c)

print("\nModel 3 — Lasso, components, CASTHMA")
m3_a, r3_a = train_and_cv('lasso', X_comp_a, y_a, grp_a, str_a, fcols_comp_a)
print("Model 3 — Lasso, components, COPD")
m3_c, r3_c = train_and_cv('lasso', X_comp_c, y_c, grp_c, str_c, fcols_comp_c)

print("\nModel 4 — Ridge, components, CASTHMA")
m4_a, r4_a = train_and_cv('ridge', X_comp_a, y_a, grp_a, str_a, fcols_comp_a)
print("Model 4 — Ridge, components, COPD")
m4_c, r4_c = train_and_cv('ridge', X_comp_c, y_c, grp_c, str_c, fcols_comp_c)

print("\nAll models trained.")

Training all 8 models (4 configs × 2 targets)...

Model 1 — Lasso, aggregate, CASTHMA
Model 1 — Lasso, aggregate, COPD

Model 2 — Ridge, aggregate, CASTHMA
Model 2 — Ridge, aggregate, COPD

Model 3 — Lasso, components, CASTHMA
Model 3 — Lasso, components, COPD

Model 4 — Ridge, components, CASTHMA
Model 4 — Ridge, components, COPD

All models trained.


In [43]:
from sklearn.linear_model import Ridge, Lasso

def cv_fold_scores(model_type, X, y, groups, strat):
    """Return per-fold R² scores using SAME CV splits."""

    cv_splits = get_cv_splits(X, strat, groups)
    scores = []

    for train_idx, val_idx in cv_splits:
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        
        if model_type == 'lasso':
            reg = Lasso(alpha=0.01, max_iter=10000, random_state=RANDOM_STATE)
            step = 'lasso'
        else:
            reg = Ridge(alpha=1.0)
            step = 'ridge'

        pipe = Pipeline([
            ('scaler', StandardScaler()),
            (step, reg)
        ])

        pipe.fit(X_tr, y_tr)
        y_pred = pipe.predict(X_val)

        scores.append(r2_score(y_val, y_pred))

    return np.array(scores)

In [44]:
# ASTHMA 
base_scores_a = cv_fold_scores('ridge', X_base_a, y_a, grp_a, str_a)
ext_scores_a  = cv_fold_scores('ridge', X_ext_a,  y_a, grp_a, str_a)

# COPD
base_scores_c = cv_fold_scores('ridge', X_base_c, y_c, grp_c, str_c)
ext_scores_c  = cv_fold_scores('ridge', X_ext_c,  y_c, grp_c, str_c)


In [45]:
BASE_FEATURES = BASE_COLS
EXT_FEATURES = BASE_COLS + PEST_AGGREGATE

In [46]:
from scipy.stats import ttest_rel

def compare_models(base_scores, ext_scores, label):
    diff = ext_scores - base_scores

    t_stat, p_value = ttest_rel(ext_scores, base_scores)

    print(f"\n===== {label} =====")
    print("Base scores:     ", np.round(base_scores, 4))
    print("Extended scores: ", np.round(ext_scores, 4))
    print("Differences:     ", np.round(diff, 4))
    print(f"Mean improvement: {diff.mean():.4f}")
    print(f"Std of diff:      {diff.std():.4f}")
    print(f"t-statistic:      {t_stat:.4f}")
    print(f"p-value:          {p_value:.4f}")

    # quick interpretation
    if p_value < 0.05:
        print("→ Statistically significant improvement from pesticides")
    else:
        print("→ No statistically significant improvement")

In [47]:
compare_models(base_scores_a, ext_scores_a, "ASTHMA (Ridge, Aggregate)")
compare_models(base_scores_c, ext_scores_c, "COPD (Ridge, Aggregate)")


===== ASTHMA (Ridge, Aggregate) =====
Base scores:      [0.6381 0.6355 0.6279 0.5976]
Extended scores:  [0.6406 0.6468 0.6346 0.6102]
Differences:      [0.0025 0.0113 0.0067 0.0126]
Mean improvement: 0.0083
Std of diff:      0.0040
t-statistic:      3.6138
p-value:          0.0364
→ Statistically significant improvement from pesticides

===== COPD (Ridge, Aggregate) =====
Base scores:      [0.8561 0.8304 0.8617 0.8555]
Extended scores:  [0.8556 0.8344 0.8651 0.8571]
Differences:      [-0.0005  0.004   0.0033  0.0016]
Mean improvement: 0.0021
Std of diff:      0.0018
t-statistic:      2.0751
p-value:          0.1296
→ No statistically significant improvement
